# Optimize spatiotemporal synthetic-data parameters (well-conditioned demo)

**Goal:** generate a *balanced* synthetic spatiotemporal paired-difference dataset where:

- IID / Cluster / Spatial engines produce tight CIs and typically conclude equivalence at Δ=1.
- SpatioTemporal engine produces **wider** (dependence-aware) CIs, **without** numerical pathologies
  (no near-singular covariance, no boundary AR(1), no extreme nugget, etc.).
- The **validation** spatiotemporal bootstrap CI (time-block bootstrap with ML refit) is **commensurate** with
  the model-based spatiotemporal CI (same qualitative width, not artificially one-sided).

This notebook performs a lightweight random search over generator parameters with explicit *sanity gates*.
It then saves the selected parameters to `best_spatiotemporal_params.json` for use in the demo notebook.


In [3]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
from pathlib import Path

REPO_ROOT = Path("/Users/dhetting/src/pyTOST")  # adjust if needed
sys.path.insert(0, str(REPO_ROOT))

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

from pyTOST import run_tost, WorkflowOptions, SpatialConfig, SpatioTemporalConfig
from pyTOST.data_gen.synthetic_tost_data import generate_spatiotemporal, EffectSpec
from pyTOST.data_gen.params_io import save_best


/Users/dhetting/src/pyTOST/pyTOST/engines/spatial_tost.py:480: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  covariance parameters :math:`(\sigma^2, \rho, \tau^2)` for each candidate ``mu``.


In [ ]:
# ---- search controls ----
SEED = 123
rng = np.random.default_rng(SEED)

N_ITER = 250          # increase if you want a stronger guarantee of finding a good candidate
MARGIN = 1.0
ALPHA = 0.05

# Bootstrap reps for *screening* (keep small so the search is fast)
B_ST_PARAM = 60       # model-based parametric bootstrap (screening)
B_ST_VALID = 60       # validation: time-block bootstrap with ML refit (screening)

# Spatiotemporal panel size (core identifiability knobs)
N_SPACE_RANGE = (30, 70)   # locations
N_TIME_RANGE  = (40, 90)   # time points

# Dependence knobs (generator)
PHI_RANGE = (0.45, 0.80)        # AR(1) rho in generator
LS_FRAC_RANGE = (0.20, 0.55)    # Matérn length-scale as fraction of domain diameter
SPATIAL_SD_RANGE = (0.4, 1.0)   # correlated component sd
OBS_SD_RANGE = (0.15, 0.60)     # nugget/obs sd

# Domain (controls "diameter" for length-scale heuristics)
DOMAIN = (-2, 2, -2, 2)
DOMAIN_DIAM = math.sqrt((DOMAIN[1]-DOMAIN[0])**2 + (DOMAIN[3]-DOMAIN[2])**2)

# Effect size (mean difference). Keep near your demo (~0.93).
DELTA_MEAN = 0.93

# ---- sanity gates (post-fit, using fitted ST params) ----
PHI_FIT_MAX = 0.90            # reject near-unit-root fits
NUGGET_FRAC_RANGE = (0.10, 0.50)
LS_FIT_FRAC_RANGE = (0.15, 0.80)   # fitted spatial length-scale as fraction of domain diameter

# CI "commensurability" gate
# Require validation width to be within a factor of [0.6, 1.6] of model-based (param boot) width.
WIDTH_RATIO_RANGE = (0.60, 1.60)

# Require ST uncertainty to be meaningfully wider than IID (avoid trivial dependence)
MIN_ST_VALID_WIDTH = 0.18


def _make_diff_st(df_long: pd.DataFrame) -> pd.DataFrame:
    """Convert long A/B rows from `generate_spatiotemporal` into a diff dataframe with (cluster,time,x,y)."""
    df = df_long.copy()
    # stable location/cluster id by coordinate
    df["building_id"] = pd.factorize(list(zip(df["x"].to_numpy(), df["y_sp"].to_numpy())))[0]
    piv = df.pivot(index=["building_id", "t", "x", "y_sp"], columns="arm", values="y").dropna()
    diff = (piv["B"] - piv["A"]).rename("diff").reset_index()
    diff = diff.rename(columns={"t": "time", "y_sp": "y"})
    return diff


def _ci_width(df_primary: pd.DataFrame) -> float:
    lo = float(df_primary.loc[df_primary["delta"] == MARGIN, "ci_low"].iloc[0])
    hi = float(df_primary.loc[df_primary["delta"] == MARGIN, "ci_high"].iloc[0])
    return float(hi - lo)


def evaluate_candidate(params: dict) -> dict:
    """Generate data, fit ST, compute model CI + validation CI, and apply sanity gates."""
    df_long, meta = generate_spatiotemporal(
        n_space=params["n_space"],
        n_time=params["n_time"],
        length_scale=params["length_scale"],
        rho=params["phi"],
        spatial_sd=params["spatial_sd"],
        obs_sd=params["obs_sd"],
        domain=DOMAIN,
        seed=int(params["seed"]),
        effect=EffectSpec(delta=DELTA_MEAN),
    )
    df = _make_diff_st(df_long)

    # ---- ST model-based CI (parametric bootstrap) ----
    cfg_param = SpatioTemporalConfig(
        mu_ci_method="parametric_bootstrap",
        mu_bootstrap_B=B_ST_PARAM,
        verbose_diagnostics=False,
        # keep defaults for regularization to reflect typical usage
    )
    out_param = run_tost(
        df,
        y="diff",
        margins=[MARGIN],
        alpha=ALPHA,
        engine="spatiotemporal",
        cluster="building_id",
        time="time",
        x="x",
        ycoord="y",
        spatiotemporal_config=cfg_param,
        options=WorkflowOptions(),
    )
    prim_param = out_param["primary"]
    w_param = _ci_width(prim_param)

    # Extract fitted covariance diagnostics (already returned by the engine)
    st_phi = float(prim_param["st_phi"].iloc[0])
    st_rho = float(prim_param["st_rho"].iloc[0])     # spatial length-scale
    st_sigma2 = float(prim_param["st_sigma2"].iloc[0])
    st_tau2 = float(prim_param["st_tau2"].iloc[0])
    nugget_frac = st_tau2 / (st_sigma2 + st_tau2 + 1e-12)

    # ---- ST validation CI: time-block bootstrap with ML refit ----
    cfg_valid = SpatioTemporalConfig(
        mu_ci_method="time_block_bootstrap",
        mu_bootstrap_B=B_ST_VALID,
        mu_timeblock_refit_cov=True,
        mu_timeblock_refit_maxiter=25,
        mu_timeblock_center=True,
        mu_timeblock_ci_style="symmetric",  # two-sided, robust for dependent bootstraps
        verbose_diagnostics=False,
    )
    out_valid = run_tost(
        df,
        y="diff",
        margins=[MARGIN],
        alpha=ALPHA,
        engine="spatiotemporal",
        cluster="building_id",
        time="time",
        x="x",
        ycoord="y",
        spatiotemporal_config=cfg_valid,
        options=WorkflowOptions(),
    )
    prim_valid = out_valid["primary"]
    w_valid = _ci_width(prim_valid)

    # ---- gates ----
    ls_fit_frac = st_rho / DOMAIN_DIAM
    ok = True
    reasons = []

    if st_phi >= PHI_FIT_MAX:
        ok = False; reasons.append(f"st_phi too high ({st_phi:.3f})")

    if not (NUGGET_FRAC_RANGE[0] <= nugget_frac <= NUGGET_FRAC_RANGE[1]):
        ok = False; reasons.append(f"nugget frac out of range ({nugget_frac:.3f})")

    if not (LS_FIT_FRAC_RANGE[0] <= ls_fit_frac <= LS_FIT_FRAC_RANGE[1]):
        ok = False; reasons.append(f"spatial ls frac out of range ({ls_fit_frac:.3f})")

    if w_valid < MIN_ST_VALID_WIDTH:
        ok = False; reasons.append(f"ST validation CI too tight (width={w_valid:.3f})")

    ratio = (w_valid / (w_param + 1e-12))
    if not (WIDTH_RATIO_RANGE[0] <= ratio <= WIDTH_RATIO_RANGE[1]):
        ok = False; reasons.append(f"width ratio out of range (valid/param={ratio:.2f})")

    return dict(
        ok=ok,
        reasons=reasons,
        params=params,
        meta=meta,
        w_param=w_param,
        w_valid=w_valid,
        width_ratio=ratio,
        st_phi=st_phi,
        st_rho=st_rho,
        nugget_frac=nugget_frac,
        ci_param=(float(prim_param.ci_low.iloc[0]), float(prim_param.ci_high.iloc[0])),
        ci_valid=(float(prim_valid.ci_low.iloc[0]), float(prim_valid.ci_high.iloc[0])),
    )


# ---- random search ----
best = None
accepted = []

for i in range(N_ITER):
    n_space = int(rng.integers(N_SPACE_RANGE[0], N_SPACE_RANGE[1] + 1))
    n_time  = int(rng.integers(N_TIME_RANGE[0],  N_TIME_RANGE[1] + 1))
    phi     = float(rng.uniform(PHI_RANGE[0], PHI_RANGE[1]))
    ls_frac = float(rng.uniform(LS_FRAC_RANGE[0], LS_FRAC_RANGE[1]))
    length_scale = float(ls_frac * DOMAIN_DIAM)
    spatial_sd = float(rng.uniform(SPATIAL_SD_RANGE[0], SPATIAL_SD_RANGE[1]))
    obs_sd    = float(rng.uniform(OBS_SD_RANGE[0], OBS_SD_RANGE[1]))

    cand = dict(
        n_space=n_space,
        n_time=n_time,
        phi=phi,
        length_scale=length_scale,
        spatial_sd=spatial_sd,
        obs_sd=obs_sd,
        seed=int(rng.integers(0, 10_000_000)),
    )

    try:
        res = evaluate_candidate(cand)
    except Exception as e:
        # ignore numerical failures; they usually correspond to bad conditioning
        continue

    if res["ok"]:
        accepted.append(res)
        # objective: make both ST CIs wide but commensurate (maximize min(widths) while keeping ratio ~ 1)
        score = min(res["w_param"], res["w_valid"]) - abs(math.log(res["width_ratio"]))
        if (best is None) or (score > best["score"]):
            best = dict(**res, score=score)

best, len(accepted)


/var/folders/mk/5nfn3tpj2jz0xqk2v9gcmv28k8pntm/T/ipykernel_35764/3872818506.py:47: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  df["building_id"] = pd.factorize(list(zip(df["x"].to_numpy(), df["y_sp"].to_numpy())))[0]
/var/folders/mk/5nfn3tpj2jz0xqk2v9gcmv28k8pntm/T/ipykernel_35764/3872818506.py:47: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  df["building_id"] = pd.factorize(list(zip(df["x"].to_numpy(), df["y_sp"].to_numpy())))[0]


In [ ]:
from pprint import pprint

if best is None:
    raise RuntimeError(
        f"No acceptable candidate found in N_ITER={N_ITER}. "
        "Increase N_ITER, relax gates, or widen parameter ranges."
    )

print("Selected parameters:")
pprint(best["params"])

print("\nST fitted diagnostics (from model-based param boot fit):")
print(f"  st_phi        = {best['st_phi']:.3f}")
print(f"  st_rho        = {best['st_rho']:.3f}  (ls frac={best['st_rho']/DOMAIN_DIAM:.3f})")
print(f"  nugget frac   = {best['nugget_frac']:.3f}")

print("\nCIs at Δ=1.0:")
print(f"  ST model (param boot): {best['ci_param']}, width={best['w_param']:.3f}")
print(f"  ST validation (block+refit): {best['ci_valid']}, width={best['w_valid']:.3f}")
print(f"  width ratio (valid/param): {best['width_ratio']:.2f}")


In [ ]:
out_path = "best_spatiotemporal_params_v2.json"

# Save only the generator parameters; these are what `generate_spatiotemporal(...)` uses.
best_params = best["params"].copy()
# Keep effect size and domain with the params so the demo is reproducible.
best_params["domain"] = DOMAIN
best_params["delta_mean"] = DELTA_MEAN

with open(out_path, "w") as f:
    json.dump(best_params, f, indent=2)

print(f"Wrote: {out_path}")


In [ ]:
# Inspect the accepted candidates (optional)
import pandas as pd

df_acc = pd.DataFrame([
    dict(
        score=r.get("score", np.nan),
        w_param=r["w_param"],
        w_valid=r["w_valid"],
        width_ratio=r["width_ratio"],
        st_phi=r["st_phi"],
        nugget_frac=r["nugget_frac"],
        ls_frac=r["st_rho"]/DOMAIN_DIAM,
        n_space=r["params"]["n_space"],
        n_time=r["params"]["n_time"],
        phi_gen=r["params"]["phi"],
        obs_sd=r["params"]["obs_sd"],
    )
    for r in accepted
])

df_acc.sort_values(["w_valid", "w_param"], ascending=False).head(10)
